# Colab CPU 离线调优

使用 Colab **免费 CPU 运行环境**：在“运行时 → 更改运行时类型”中选择 Python 3，硬件加速器选“无”。
本 notebook（交互式笔记本）只调度已冻结的 Rust 候选，使用纯合成样本；GPU 和付费运行环境均不需要。
免费资源与会话时长由 Colab 决定，6 小时是实验预算上限，运行环境可能提前断开。

**上传前在本机审阅源码包。** ZIP 含 Rust 源码、默认业务配置、测试基线和候选补丁；
默认配置可能含业务标识，必须一起审阅。只上传已获准的 bundle ZIP。
**禁止上传真实 ERP 数据、真实处理结果、凭据或本地原始日志。** 本机保存 `bundle-review.json`，
从该文件独立取得 `archive_sha256`；不要从待上传 ZIP 内取得预期哈希。

所有安装和实验文件写入本次 Colab 虚拟机的 `/content` 独立目录。
不自动挂载或写入 Google Drive，不上传 GitHub。联网仅用于下载固定版本工具链和锁定依赖；
运行 driver（实验驱动程序）后只处理合成数据。云端排名用于筛选，采纳前仍需目标电脑真实 GB/SK 验收。
按顺序运行下列单元。重新开始整个实验时，从“本地目录”单元建立新的会话目录。


In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

UV_VERSION = '0.13.6'
PYTHON_VERSION = '3.13.18'  # 项目支持 Python >=3.11；固定完整版本以复现实验。
RUST_VERSION = '1.96.0'
RUSTUP_VERSION = '1.28.2'


def run_environment(session: Path) -> dict[str, str]:
    env = {
        key: value
        for key, value in os.environ.items()
        if not key.startswith(('UV_', 'PIP_', 'CARGO_', 'RUSTUP_'))
        and key not in {'PYTHONPATH', 'PYTHONHOME', 'RUSTFLAGS', 'RUSTC_WRAPPER'}
    }
    env.update(
        PATH=f'{session}/cargo/bin:{session}/bootstrap/bin:/usr/local/bin:/usr/bin:/bin',
        CARGO_HOME=str(session / 'cargo'),
        RUSTUP_HOME=str(session / 'rustup'),
        RUSTUP_TOOLCHAIN=RUST_VERSION,
        UV_PROJECT_ENVIRONMENT=str(session / 'venv'),
        UV_PYTHON_INSTALL_DIR=str(session / 'python'),
        UV_CACHE_DIR=str(session / 'uv-cache'),
        UV_TOOL_DIR=str(session / 'uv-tools'),
        UV_TOOL_BIN_DIR=str(session / 'uv-bin'),
        PYTHONDONTWRITEBYTECODE='1',
        PYTHONNOUSERSITE='1',
        TMPDIR=str(session / 'tmp'),
    )
    return env


def command_plan(session: Path, bundle: Path, work: Path) -> dict[str, list[str]]:
    bootstrap_python = str(session / 'bootstrap/bin/python')
    uv = str(session / 'bootstrap/bin/uv')
    return {
        'bootstrap': [sys.executable, '-I', '-m', 'venv', str(session / 'bootstrap')],
        'install-uv': [
            bootstrap_python,
            '-I',
            '-m',
            'pip',
            '--isolated',
            'install',
            '--disable-pip-version-check',
            '--no-cache-dir',
            '--no-deps',
            '--only-binary=:all:',
            '--index-url',
            'https://pypi.org/simple',
            f'uv=={UV_VERSION}',
        ],
        'install-python': [uv, 'python', 'install', '--no-bin', PYTHON_VERSION],
        'sync': [uv, 'sync', '--frozen', '--extra', 'dev', '--python', PYTHON_VERSION, '--managed-python'],
        'install-rustup': [
            str(session / 'downloads/rustup-init'),
            '-y',
            '--no-modify-path',
            '--profile',
            'minimal',
            '--default-toolchain',
            'none',
        ],
        'install-rust': [
            str(session / 'cargo/bin/rustup'),
            'toolchain',
            'install',
            RUST_VERSION,
            '--profile',
            'minimal',
            '--component',
            'rustfmt',
            '--component',
            'clippy',
        ],
        'active-toolchain': [str(session / 'cargo/bin/rustup'), 'show', 'active-toolchain'],
        'cargo-version': [str(session / 'cargo/bin/cargo'), '--version', '--verbose'],
        'driver': [
            str(session / 'venv/bin/python'),
            '-m',
            'tools.validation.autotune_experiment',
            '--bundle-root',
            str(bundle),
            '--work-dir',
            str(work),
            '--rust-tests',
            '--pairs',
            '8',
            '--calibration-pairs',
            '4',
            '--budget-seconds',
            '21600',
        ],
    }

## 1. 本地目录

在全新的 Colab CPU 会话中运行。本单元创建专属目录，打印 ZIP 上传位置。
控制代码、上传文件、解压源码、工具安装和实验输出分别存放；Python 读取外部文件时使用 `-I`
（隔离模式：忽略当前目录和用户 Python 路径），避免同名下载文件被误导入。


In [ ]:
if platform.system() != 'Linux' or platform.machine() != 'x86_64' or not Path('/content').is_dir():
    raise RuntimeError('请选择 Colab Linux x86_64 的免费 CPU 运行环境。')

SESSION_ROOT = Path(tempfile.mkdtemp(prefix='costing-autotune-', dir='/content'))
CONTROL_DIR = SESSION_ROOT / 'control'
INCOMING_DIR = SESSION_ROOT / 'incoming'
for directory in (CONTROL_DIR, INCOMING_DIR, SESSION_ROOT / 'tmp'):
    directory.mkdir()
os.chdir(CONTROL_DIR)
RUN_ENV = run_environment(SESSION_ROOT)
for tool in ('git', 'patch', 'cc'):
    if shutil.which(tool, path=RUN_ENV['PATH']) is None:
        raise RuntimeError(f'运行环境缺少 {tool}；请选择标准 Colab CPU 环境。')

BUNDLE_ROOT = SESSION_ROOT / 'bundle'
SOURCE_ROOT = BUNDLE_ROOT / 'source'
WORK_DIR = SESSION_ROOT / 'experiment-01'  # driver 要求尚不存在的目录；不要提前 mkdir。
COMMANDS = command_plan(SESSION_ROOT, BUNDLE_ROOT, WORK_DIR)
print(f'本次目录：{SESSION_ROOT}')
print(f'请通过 Colab 左侧“文件”面板，仅将已审阅的 ZIP 上传到：{INCOMING_DIR}')

In [ ]:
SAFE_EXTRACT_SCRIPT = r"""
import hashlib
import json
import re
import stat
import sys
from pathlib import Path, PurePosixPath, PureWindowsPath
from zipfile import ZIP_DEFLATED, ZIP_STORED, BadZipFile, ZipFile

MAX_ARCHIVE_BYTES = 128 * 1024**2
MAX_FILE_BYTES = 32 * 1024**2
MAX_TOTAL_BYTES = 256 * 1024**2
MAX_MEMBERS = 20000
MAX_RATIO = 200


def safe_extract_bundle(archive_path: Path, expected_sha256: str, destination: Path) -> dict:
    expected = expected_sha256.strip().lower()
    if re.fullmatch(r'[0-9a-f]{64}', expected) is None:
        raise ValueError('预期 SHA-256 必须为 64 位十六进制字符。')
    if destination.exists() or destination.is_symlink():
        raise ValueError('解压目录必须是尚不存在的新目录。')
    if archive_path.is_symlink() or not archive_path.is_file():
        raise ValueError('ZIP 必须是普通文件。')
    # 同一文件句柄完成哈希核对和解压，避免两次打开之间替换路径。
    with archive_path.open('rb') as raw:
        raw.seek(0, 2)
        if raw.tell() > MAX_ARCHIVE_BYTES:
            raise ValueError('ZIP 压缩文件超过大小上限。')
        raw.seek(0)
        actual = hashlib.file_digest(raw, 'sha256').hexdigest()
        if actual != expected:
            raise ValueError('ZIP SHA-256 与本机审阅记录不匹配。')
        raw.seek(0)
        with ZipFile(raw) as archive:
            members = archive.infolist()
            if not members or len(members) > MAX_MEMBERS:
                raise ValueError('ZIP 条目数量超限或为空。')
            paths = set()
            file_names = set()
            total = 0
            for member in members:
                name = member.filename
                clean = name[:-1] if member.is_dir() else name
                parts = clean.split('/')
                if (
                    not clean or '\\' in name or '\x00' in member.orig_filename
                    or PurePosixPath(clean).is_absolute() or PureWindowsPath(clean).drive
                    or ':' in clean or any(part in {'', '.', '..'} for part in parts)
                ):
                    raise ValueError('ZIP 含不安全路径。')
                if clean in paths:
                    raise ValueError('ZIP 含重复项。')
                paths.add(clean)
                kind = stat.S_IFMT(member.external_attr >> 16)
                if kind not in {0, stat.S_IFREG, stat.S_IFDIR}:
                    raise ValueError('ZIP 含符号链接或特殊文件。')
                if kind == stat.S_IFDIR and not member.is_dir():
                    raise ValueError('ZIP 目录类型与路径不一致。')
                if member.flag_bits & 1 or member.compress_type not in {ZIP_STORED, ZIP_DEFLATED}:
                    raise ValueError('ZIP 含加密项或不支持的压缩方式。')
                if member.is_dir():
                    if member.file_size:
                        raise ValueError('ZIP 目录项含文件内容。')
                else:
                    file_names.add(clean)
                total += member.file_size
                if member.file_size > MAX_FILE_BYTES or total > MAX_TOTAL_BYTES:
                    raise ValueError('ZIP 解压大小超过上限。')
                if member.file_size > max(1, member.compress_size) * MAX_RATIO:
                    raise ValueError('ZIP 压缩膨胀比例超过上限。')
            if any(str(parent) in file_names for name in paths for parent in PurePosixPath(name).parents):
                raise ValueError('ZIP 文件与目录路径冲突。')
            required = {
                'bundle-manifest.json', 'source/pyproject.toml', 'source/uv.lock',
                'source/rust-toolchain.toml', 'source/rust/Cargo.toml', 'source/rust/Cargo.lock',
                'source/tools/validation/autotune_experiment.py',
            }
            if not required <= file_names:
                raise ValueError('ZIP 缺少 bundle 必需文件。')
            destination.mkdir(mode=0o700)
            written_total = 0
            for member in members:
                target = destination.joinpath(*PurePosixPath(member.filename).parts)
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                written = 0
                with archive.open(member) as source, target.open('xb') as output:
                    while block := source.read(1024 * 1024):
                        written += len(block)
                        written_total += len(block)
                        if written > min(member.file_size, MAX_FILE_BYTES) or written_total > MAX_TOTAL_BYTES:
                            raise ValueError('ZIP 实际解压大小超过上限。')
                        output.write(block)
                if written != member.file_size:
                    raise ValueError('ZIP 实际文件大小不匹配。')
    return {'bundle_root': str(destination), 'archive_sha256': actual, 'file_count': len(file_names)}


if __name__ == '__main__':
    try:
        result = safe_extract_bundle(Path(sys.argv[1]), sys.argv[2], Path(sys.argv[3]))
    except (OSError, ValueError, BadZipFile) as error:
        print(json.dumps({'error_code': 'BUNDLE_REJECTED', 'message': str(error), 'retryable': False}))
        raise SystemExit(2) from None
    print(json.dumps(result))
"""

## 2. 手动核对并解压

确认本机审阅完成后，通过文件面板上传 ZIP，在下面输入其绝对路径和独立保存的 `archive_sha256`。
不要上传真实 ERP。程序会重新计算整个 ZIP 的 SHA-256，通过后才检查和解压。

限制：ZIP 最多 128 MiB（兆字节），单文件最多 32 MiB，解压总计最多 256 MiB、20,000 个条目，
单项解压膨胀比例最多 200 倍。重复项、绝对路径、`..`、反斜杠、符号链接和文件/目录冲突均拒绝。
超限时先在本机审查 bundle 内容。解压失败后的目录保留用于检查；重新开始请建立新会话目录。


In [ ]:
ZIP_PATH = Path(input('已审阅 ZIP 的绝对路径：').strip())
EXPECTED_SHA256 = input('本机 bundle-review.json 中的 archive_sha256：').strip()
if not ZIP_PATH.is_absolute() or ZIP_PATH.parent.resolve() != INCOMING_DIR.resolve():
    raise ValueError(f'请仅使用本次上传目录中的 ZIP：{INCOMING_DIR}')
subprocess.run(  # noqa: S603 - 固定命令或已核验脚本，显式参数且不使用 shell。
    [sys.executable, '-I', '-c', SAFE_EXTRACT_SCRIPT, str(ZIP_PATH), EXPECTED_SHA256, str(BUNDLE_ROOT)],
    cwd=CONTROL_DIR,
    env=RUN_ENV,
    check=True,
    timeout=120,
)

## 3. 固定 Python 验证环境

uv 是 Python 环境管理工具，只安装到本次 Colab 的 `bootstrap` 目录，不加入项目生产依赖。
固定 uv 0.13.6、Python 3.13.18，使用已审阅源码中的 `uv.lock` 执行 `uv sync --frozen --extra dev`。
同步命令从 `source` 运行；环境和缓存位于源码树外，禁止生成 `.pyc` 字节码，保持 bundle 树干净。
此步骤需要联网下载已锁定依赖。失败会直接停止当前单元。


In [ ]:
for step in ('bootstrap', 'install-uv', 'install-python'):
    subprocess.run(  # noqa: S603 - 固定的安装与工具链命令，显式参数且不使用 shell。
        COMMANDS[step], cwd=CONTROL_DIR, env=RUN_ENV, check=True, timeout=600
    )
subprocess.run(  # noqa: S603 - 从已核对哈希的源码目录同步锁定依赖。
    COMMANDS['sync'], cwd=SOURCE_ROOT, env=RUN_ENV, check=True, timeout=900
)
subprocess.run(  # noqa: S603 - 固定命令或已核验脚本，显式参数且不使用 shell。
    [str(SESSION_ROOT / 'venv/bin/python'), '-I', '--version'],
    cwd=CONTROL_DIR,
    env=RUN_ENV,
    check=True,
    timeout=30,
)

In [ ]:
RUSTUP_DOWNLOAD_SCRIPT = r"""
import hashlib
import json
import re
import stat
import sys
import tomllib
import urllib.request
from pathlib import Path


def download_rustup(download_dir: Path, source_root: Path, version: str) -> Path:
    channel = tomllib.loads((source_root / 'rust-toolchain.toml').read_text(encoding='utf-8'))['toolchain']['channel']
    if channel != '1.96.0':
        raise ValueError('bundle 的 Rust 工具链必须固定为 1.96.0。')
    if re.fullmatch(r'\d+\.\d+\.\d+', version) is None:
        raise ValueError('rustup 版本格式无效。')
    url = f'https://static.rust-lang.org/rustup/archive/{version}/x86_64-unknown-linux-gnu/rustup-init'
    with urllib.request.urlopen(url + '.sha256', timeout=60) as response:
        checksum = response.read(4097).decode('ascii')
    match = re.fullmatch(r'([0-9a-f]{64})(?:\s+\*?(?:\./)?rustup-init)?\s*', checksum)
    if len(checksum) > 4096 or match is None:
        raise ValueError('官方 rustup-init SHA-256 文件格式无效。')
    download_dir.mkdir(mode=0o700)
    installer = download_dir / 'rustup-init'
    digest = hashlib.sha256()
    size = 0
    with urllib.request.urlopen(url, timeout=60) as response, installer.open('xb') as output:
        while block := response.read(1024 * 1024):
            size += len(block)
            if size > 64 * 1024**2:
                raise ValueError('rustup-init 下载超过大小上限。')
            digest.update(block)
            output.write(block)
    if digest.hexdigest() != match[1]:
        raise ValueError('rustup-init 官方 SHA-256 校验失败。')
    installer.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
    print(json.dumps({'rustup_version': version, 'installer_sha256': digest.hexdigest()}))
    return installer


if __name__ == '__main__':
    download_rustup(Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3])
"""

## 4. 固定 Rust 工具链

从 Rust 官方 `static.rust-lang.org` 下载固定版 rustup-init 1.28.2 和同站 SHA-256 文件，
核验通过后才赋予执行权限并运行。这里校验下载完整性，信任来源是官方 HTTPS 站点及其校验文件。
安装目标 `CARGO_HOME`、`RUSTUP_HOME` 均为本次 Colab 本地目录；`--no-modify-path` 不修改 shell 配置。
子进程显式传递 `PATH`，Rust 固定为 1.96.0。cargo 在 `source` 或 driver 的独立源码树内运行，
使根目录 `rust-toolchain.toml` 生效；下方会打印实际工具链和 cargo 版本。


In [ ]:
subprocess.run(  # noqa: S603 - 固定命令或已核验脚本，显式参数且不使用 shell。
    [
        sys.executable,
        '-I',
        '-c',
        RUSTUP_DOWNLOAD_SCRIPT,
        str(SESSION_ROOT / 'downloads'),
        str(SOURCE_ROOT),
        RUSTUP_VERSION,
    ],
    cwd=CONTROL_DIR,
    env=RUN_ENV,
    check=True,
    timeout=300,
)
for step in ('install-rustup', 'install-rust', 'active-toolchain', 'cargo-version'):
    subprocess.run(  # noqa: S603 - 固定的安装与工具链命令，显式参数且不使用 shell。
        COMMANDS[step], cwd=SOURCE_ROOT, env=RUN_ENV, check=True, timeout=900
    )

## 5. 执行合成实验

driver 从 `source` 目录运行，`--bundle-root` 指向解压根目录，`--work-dir` 指向独立且尚不存在的目录。
启用 `--rust-tests`；正式测量固定 8 对、校准固定 4 对、总墙钟预算 21,600 秒。
由 driver 顺序构建、校验和测量，请等待它结束后再执行后续单元，测速期间不要运行其他计算任务。

命令失败仍可在后续单元检查已保存报告与日志。噪声不合格、无候选入围或预算耗尽都有各自状态，
请结合报告判断；运行完成本身不表示候选已获采纳。


In [ ]:
if WORK_DIR.exists():
    raise RuntimeError('整个实验需要新的工作目录；请重新运行“本地目录”单元并重新校准。')
print('实验工作目录：', WORK_DIR)
subprocess.run(  # noqa: S603 - 使用已审阅 bundle 的 driver，参数显式传递。
    COMMANDS['driver'], cwd=SOURCE_ROOT, env=RUN_ENV, check=True
)

In [ ]:
REPORT_PATH = WORK_DIR / 'experiment-report.json'
if not REPORT_PATH.is_file():
    raise RuntimeError(f'尚无实验报告，请检查上方错误和 {WORK_DIR / "logs"}。')
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(
    json.dumps(
        {
            'status': report.get('status'),
            'shortlist': report.get('shortlist', []),
            'recommended': report.get('recommended', []),
            'error': report.get('error'),
            'candidates': {
                name: {
                    'status': entry.get('status'),
                    'summary': (entry.get('primary') or {}).get('summary'),
                }
                for name, entry in report.get('candidates', {}).items()
            },
        },
        ensure_ascii=False,
        indent=2,
    )
)

## 6. 下载审阅材料

下方只打包合成实验报告、配对测量报告、候选补丁、bundle manifest（文件清单）与构建/测试日志。
工作簿、可执行文件、源码和 Python 环境均留在本次虚拟机。
先检查报告与日志，再手动运行下载单元保存到自己的电脑；Colab 断开后本地磁盘可能丢失。
材料内可能含已审阅的候选源码片段和本次虚拟机路径，对外分享前仍需审阅。


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile


def make_results_archive(bundle: Path, work: Path, output: Path) -> Path:
    manifest_path = bundle / 'bundle-manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    selected = {
        'bundle-manifest.json': (bundle, manifest_path),
        'experiment-report.json': (work, work / 'experiment-report.json'),
    }
    for group, pattern in (
        ('logs', '*.log'),
        ('measurements', 'report.json'),
        ('comparisons', '*.json'),
        ('fixtures', 'spec.json'),
    ):
        for path in (work / group).rglob(pattern):
            selected[path.relative_to(work).as_posix()] = (work, path)
    for candidate in manifest['candidates']:
        relative = candidate['patch']
        selected[relative] = (bundle, bundle / relative)
    for name, (root, path) in selected.items():
        if (
            path.is_symlink()
            or not path.is_file()
            or not path.resolve().is_relative_to(root.resolve())
            or name.startswith('/')
            or '..' in Path(name).parts
        ):
            raise ValueError('下载材料含缺失或不安全的文件。')
    if sum(path.stat().st_size for _, path in selected.values()) > 128 * 1024**2:
        raise ValueError('下载材料超过 128 MiB；请先审阅本地日志体积。')
    with ZipFile(output, 'x', compression=ZIP_DEFLATED) as archive:
        for name, (_, path) in sorted(selected.items()):
            archive.write(path, name)
    return output

In [ ]:
RESULTS_ZIP = make_results_archive(BUNDLE_ROOT, WORK_DIR, SESSION_ROOT / 'synthetic-autotune-results.zip')
print(f'可下载材料：{RESULTS_ZIP}（{RESULTS_ZIP.stat().st_size} 字节）')

In [ ]:
from google.colab import files

files.download(str(RESULTS_ZIP))

## 7. 中断与恢复

测量级同会话恢复入口已经存在：`python -m tools.validation.measure_paired_release ... --resume`。
仅对同一虚拟机、同一会话中未完成的单项测量使用它；保持原有 binary（可执行文件）、输入、配置、
测量目录及参数一致，从 `source` 目录、同一 `RUN_ENV` 和验证环境 Python 运行。
`--resume` 属于测量工具，整个实验 driver 不接受这个参数；本 notebook 不自动拼接恢复命令。

整个实验开始新会话时，建立新目录并重新校准，当前候选完整重跑 8 对。
不同会话的旧报告只作独立记录，不合并计时样本。下载的报告与补丁用于回到目标电脑复测，
真实数据验收始终在本地完成。
